# Tutorial 6: Reaction Enumeration and Scaffold Analysis

Use reaction SMARTS to build small virtual libraries and inspect their core scaffolds.


## Objectives

- Define a simple condensation reaction with RDKit SMARTS notation.
- Enumerate the combinatorial products from small reactant sets.
- Extract Bemis–Murcko scaffolds from the enumerated products to understand their shared cores.


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
import pandas as pd


## Define the Reaction Template

We will model a classic amide formation between an acid chloride and an aniline derivative.


In [ ]:
reaction_smarts = "[Cl:1][C:2](=O)[C:3].[NH2:4][c:5]1cccc[c:6]1>>[NH:4][C:2](=O)[c:5]1cccc[c:6]1"
rxn = AllChem.ReactionFromSmarts(reaction_smarts)
rxn


## Enumerate the Products

Provide small lists of reactants and use `EnumerateLibraryFromReaction` to generate the virtual products.


In [ ]:
acyl_chlorides = [
    Chem.MolFromSmiles('ClC(=O)C'),
    Chem.MolFromSmiles('ClC(=O)Cc1ccccc1'),
]
anilines = [
    Chem.MolFromSmiles('Nc1ccccc1'),
    Chem.MolFromSmiles('Nc1ccc(O)cc1'),
    Chem.MolFromSmiles('Nc1ccc(Cl)cc1'),
]
product_sets = list(AllChem.EnumerateLibraryFromReaction(rxn, [acyl_chlorides, anilines]))
products = [prod[0] for prod in product_sets]
product_smiles = [Chem.MolToSmiles(p) for p in products]
pd.DataFrame({'product_smiles': product_smiles})


## Extract Bemis–Murcko Scaffolds

The Murcko scaffold keeps the ring systems and linkers common to each product, revealing the shared core chemistry.


In [ ]:
scaffold_map = {}
for smiles, mol in zip(product_smiles, products):
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    scaffold_map.setdefault(Chem.MolToSmiles(scaffold), set()).add(smiles)
scaffold_df = pd.DataFrame(
    [{'scaffold_smiles': scaffold, 'example_products': sorted(list(examples))} for scaffold, examples in scaffold_map.items()]
)
scaffold_df


## Visualise Unique Scaffolds

Rendering the scaffolds highlights the dominant ring systems created by the enumeration.


In [ ]:
scaffolds = [Chem.MolFromSmiles(s) for s in scaffold_df['scaffold_smiles']]
Draw.MolsToGridImage(scaffolds, legends=[f"Scaffold {i+1}" for i in range(len(scaffolds))])
